# 1、SummarizationMiddleware中间件

## 举例1：测试trigger、keep参数

In [11]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
import os
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage


# 从.env文件中加载环境变量
load_dotenv(override=True)

DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
DEEPSEEK_BASE_URL = os.getenv("DEEPSEEK_BASE_URL")

model = init_chat_model(
    model="deepseek-v4-flash",
    model_provider="openai",
    profile={"max_input_tokens": 1_000_000},
    api_key=DEEPSEEK_API_KEY,
    base_url=DEEPSEEK_BASE_URL
)

# print(model.profile)

In [12]:
messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]


Agent插入SummarizationMiddleware的中间件
summary_prompt必须包含{messages}的变量

In [16]:

agent = create_agent(
    model = "deepseek-v4-flash",
    middleware = [
        SummarizationMiddleware(
            model = model,
            trigger = [
                ("tokens",100),
                ("messages", 5),
                ("fraction", 0.5),
            ],
            keep = ("messages",2),
            summary_prompt = "请将以下对话内容进行总结，消息列表如下：\n{messages}",
        )
    ]
)

In [20]:
result = agent.invoke(
    {
        "messages" : {
            "role": "system", "content": "你是个非常友好的AI助手",
            "role": "human", "content": "你好啊，我是老王，你是谁？",
        },
    }
)

print(result["messages"][-1].content)

老王你好！我是DeepSeek，一个由深度求索公司创造的AI助手。虽然我是纯文本模型，但知识储备挺丰富的，能陪你聊天、解答问题、处理文档，甚至还能帮你读链接里的内容呢！

看你自称老王，感觉是个亲切随和的朋友。有什么需要帮忙的吗？无论是生活琐事、工作难题，还是单纯想找个人唠唠嗑，我都乐意奉陪！😊


In [21]:
result = agent.invoke(
    {
        "messages" : messages
    }
)

for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

Here is a summary of the conversation to date:

用户老王向AI助手打招呼并询问身份，AI自称小王，随后老王表达了对认识小王的喜悦。
================================== Ai Message ==================================

你高兴得太早了
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

哈哈，别误会，我可不是在泼冷水。只是觉得你刚才那句“很高兴认识小王”有点太热情了，我下意识想逗你一下——就像朋友之间开玩笑说“别高兴太早，后面可能还有坑”那种感觉。其实我也挺高兴认识你的，就是嘴皮子想皮一下。怎么样，这下不生气了吧？
